<a href="https://colab.research.google.com/github/hathaall/GlomoAI/blob/main/GlomoAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
import os
import cv2
import numpy as np

image_dir = '/tmp/‏‏thyroid_dataset/' # Corrected path, using the exact string from 'ls -R /tmp/' output

print(f"Checking base image directory: {image_dir}")
if not os.path.exists(image_dir):
    print(f"Error: Base image directory {image_dir} does not exist.")

# Directly construct paths to subdirectories
cancerous_dir = os.path.join(image_dir, 'Cancerous')
non_cancerous_dir = os.path.join(image_dir, 'non_Cancerous')

print(f"Constructed cancerous_dir: {cancerous_dir}")
print(f"Constructed non_cancerous_dir: {non_cancerous_dir}")

cancerous_images = []
non_cancerous_images = []
labels = []

# Load cancerous images
if os.path.exists(cancerous_dir):
    for filename in os.listdir(cancerous_dir):
        img_path = os.path.join(cancerous_dir, filename)
        img = cv2.imread(img_path)
        if img is not None:
            cancerous_images.append(img)
            labels.append(1) # 1 for cancerous
    print(f"Loaded {len(cancerous_images)} cancerous images.")
else:
    print(f"Cancerous image directory '{cancerous_dir}' not found!")

# Load non-cancerous images
if os.path.exists(non_cancerous_dir):
    for filename in os.listdir(non_cancerous_dir):
        img_path = os.path.join(non_cancerous_dir, filename)
        img = cv2.imread(img_path)
        if img is not None:
            non_cancerous_images.append(img)
            labels.append(0) # 0 for non-cancerous
    print(f"Loaded {len(non_cancerous_images)} non-cancerous images.")
else:
    print(f"Non-cancerous image directory '{non_cancerous_dir}' not found!")

# Combine the image data and labels
all_images = cancerous_images + non_cancerous_images
labels = np.array(labels)

print(f"Total images loaded: {len(all_images)}")
print(f"Total labels created: {len(labels)}")

Checking base image directory: /tmp/‏‏thyroid_dataset/
Constructed cancerous_dir: /tmp/‏‏thyroid_dataset/Cancerous
Constructed non_cancerous_dir: /tmp/‏‏thyroid_dataset/non_Cancerous
Loaded 10 cancerous images.
Loaded 10 non-cancerous images.
Total images loaded: 20
Total labels created: 20


### Uploading and Unzipping the Dataset

First, upload your zipped dataset folder using the files panel on the left or `files.upload()`.

Then, use the following code to unzip it. Make sure the unzipped directory structure matches what the original notebook expects (e.g., `Thyroid/cancerous` and `Thyroid/non_cancerous` inside the unzipped folder).

In [35]:
import cv2
import numpy as np

# Define the desired image size
IMG_WIDTH = 128
IMG_HEIGHT = 128

processed_images = []

# Ensure all_images is not empty before processing
if not all_images:
    raise ValueError("Error: all_images is empty. Images were not loaded! Make sure your dataset is uploaded and unzipped.")
else:
    for img in all_images:
        # Resize the image
        resized_img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
        # Normalize pixel values
        normalized_img = resized_img / 255.0
        processed_images.append(normalized_img)

processed_images = np.array(processed_images, dtype=np.float32)
labels = np.array(labels, dtype=np.int32)

print(f"Processed images shape: {processed_images.shape}")
print(f"Labels shape: {labels.shape}")

Processed images shape: (20, 128, 128, 3)
Labels shape: (20,)


In [36]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Build the model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_WIDTH, IMG_HEIGHT, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Sigmoid for binary classification
])

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [37]:
# Train the model
# We will use a small number of epochs for demonstration purposes.
# You may need to adjust the number of epochs and batch size for better performance.
history = model.fit(processed_images, labels, epochs=10, batch_size=4)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 216ms/step - accuracy: 0.4000 - loss: 1.3492
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 157ms/step - accuracy: 0.5500 - loss: 0.6961
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 153ms/step - accuracy: 0.6000 - loss: 0.6328
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 160ms/step - accuracy: 0.5500 - loss: 0.7503
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 150ms/step - accuracy: 0.6000 - loss: 0.6630
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 160ms/step - accuracy: 0.6000 - loss: 0.6824
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 159ms/step - accuracy: 0.8500 - loss: 0.6236
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 160ms/step - accuracy: 0.6500 - loss: 0.5509
Epoch 9/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 161ms/step - accuracy: 0.8000 - loss: 0.4203
Epoch 10/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 155ms/step - accuracy: 0.8000 - loss: 0.5889


In [38]:
# Evaluate the model (optional, but good practice)
loss, accuracy = model.evaluate(processed_images, labels)

print(f"Model loss: {loss}")
print(f"Model accuracy: {accuracy}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 390ms/step - accuracy: 0.7500 - loss: 0.6156
Model loss: 0.6155945062637329
Model accuracy: 0.75


In [39]:
from google.colab import files
from tensorflow.keras.preprocessing import image

# Define the desired image size
IMG_WIDTH = 128
IMG_HEIGHT = 128

uploaded = files.upload()

for fn in uploaded.keys():
    # Load and preprocess the image
    img_path = fn
    img = cv2.imread(img_path)
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    img = img / 255.0
    img = np.expand_dims(img, axis=0) # Add batch dimension

    # Make a prediction
    prediction = model.predict(img)

    # Interpret the prediction
    if prediction[0][0] > 0.5:
        print(f"The image '{fn}' is predicted to be cancerous.")
    else:
        print(f"The image '{fn}' is predicted to be non-cancerous.")

    # Display prediction score as a percentage
    print(f"Prediction score: {prediction[0][0]*100:.2f}%")

Saving IMG_6914_png.rf.8d748e28a865a39ecd8f043068345f7d.jpg to IMG_6914_png.rf.8d748e28a865a39ecd8f043068345f7d (3).jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
The image 'IMG_6914_png.rf.8d748e28a865a39ecd8f043068345f7d (3).jpg' is predicted to be cancerous.
Prediction score: 97.55%
